GSE - algoritmo per la ricostruzione spaziale di grafi bipartiti
in optimOps.py la principale indiziata e' la funzione a riga 1539

c'e' l'estrazione degli indici e li ridefinisce con np.unique per avere una successione senza buchi.
Forse qui bisogna rimaneggiare

In [ ]:
def reindex_input_files(path):
    
    if sysOps.check_file_exists('link_assoc.npy',path):
        data = np.load(path + 'link_assoc.npy')
    elif sysOps.check_file_exists('link_assoc.txt',path):
        data = np.loadtxt(path + 'link_assoc.txt', delimiter=',', dtype=np.float64)[:,1:]
    else:
        raise ValueError("Unsupported file format")

    # Extract type 1 and type 2 indices
    type1_indices = data[:, 0].astype(int)
    type2_indices = data[:, 1].astype(int)

    # Find unique indices for type 1 and type 2
    unique_type1, reindexed_type1 = np.unique(type1_indices, return_inverse=True)
    unique_type2, reindexed_type2 = np.unique(type2_indices, return_inverse=True)

    # Reindex type 2 to start after type 1 indices
    reindexed_type2 += len(unique_type1)

    # Combine the reindexed indices into the final matrix
    reindexed_data = np.column_stack((reindexed_type1, reindexed_type2, data[:, 2])).astype(np.float64)

    # Save the reindexed matrix
    np.save(path + "link_assoc_reindexed.npy", reindexed_data)

    # Create index_key array
    type1_key = np.column_stack((np.zeros_like(unique_type1, dtype=np.float64), unique_type1, np.arange(len(unique_type1))))
    type2_key = np.column_stack((np.ones_like(unique_type2, dtype=np.float64), unique_type2, np.arange(len(unique_type1),len(unique_type1)+len(unique_type2))))
    index_key = np.vstack((type1_key, type2_key)).astype(np.int32)

    # Save the index_key array
    np.save(path + "index_key.npy", index_key)
    return len(unique_type1), len(unique_type2)